In [ ]:
text = """
artificial intelligence is transforming modern society.
it is used in healthcare finance education and transportation.
machine learning allows systems to improve automatically with experience.
data plays a critical role in training intelligent systems.
large datasets help models learn complex patterns.
deep learning uses multi layer neural networks.
neural networks are inspired by biological neurons.
each neuron processes input and produces an output.
training a neural network requires optimization techniques.
gradient descent minimizes the loss function.
"""


In [ ]:
import torch
import torch.nn as nn

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {c:i for i,c in enumerate(chars)}
idx_to_char = {i:c for i,c in enumerate(chars)}


In [ ]:
seq_len = 40
X, y = [], []

for i in range(len(text) - seq_len):
    X.append([char_to_idx[c] for c in text[i:i+seq_len]])
    y.append(char_to_idx[text[i+seq_len]])

X = torch.tensor(X)
y = torch.tensor(y)


In [ ]:
# LSTM :DDDDDDD

In [ ]:
class TextLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
model = TextLSTM(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for epoch in range(30):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 3.3400
Epoch 5, Loss: 2.9050
Epoch 10, Loss: 2.6869
Epoch 15, Loss: 2.4815
Epoch 20, Loss: 2.2543
Epoch 25, Loss: 2.0207


In [ ]:
model = TextLSTM(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for epoch in range(30):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 3.3267
Epoch 5, Loss: 2.9083
Epoch 10, Loss: 2.6510
Epoch 15, Loss: 2.4380
Epoch 20, Loss: 2.2123
Epoch 25, Loss: 1.9873


In [ ]:
def generate_text(model, seed, length=200):
    model.eval()
    generated = seed

    for _ in range(length):
        seq = torch.tensor([[char_to_idx[c] for c in generated[-seq_len:]]])
        with torch.no_grad():
            output = model(seq)
        idx = torch.argmax(output, dim=1).item()
        generated += idx_to_char[idx]

    return generated

print(generate_text(model, "artificial what", 300))


artificial whatin al neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran neuran 


In [ ]:
# Transformer :D

In [ ]:
words = text.lower().split()
vocab = sorted(set(words))
vocab_size = len(vocab)

word_to_idx = {w:i for i,w in enumerate(vocab)}
idx_to_word = {i:w for i,w in enumerate(vocab)}


In [ ]:
seq_len = 6
X, y = [], []

for i in range(len(words) - seq_len):
    X.append([word_to_idx[w] for w in words[i:i+seq_len]])
    y.append(word_to_idx[words[i+seq_len]])

X = torch.tensor(X)
y = torch.tensor(y)


In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = self.pos(x)
        x = self.encoder(x)
        return self.fc(x[:, -1])


In [ ]:
model = TransformerLM(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(40):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 4.3041
Epoch 5, Loss: 1.3469
Epoch 10, Loss: 0.4983
Epoch 15, Loss: 0.3357
Epoch 20, Loss: 0.2680
Epoch 25, Loss: 0.2288
Epoch 30, Loss: 0.2080
Epoch 35, Loss: 0.2007


In [ ]:
def generate_words(model, seed_words, length=30):
    model.eval()
    generated = seed_words.copy()

    for _ in range(length):
        seq = torch.tensor([[word_to_idx[w] for w in generated[-seq_len:]]])
        with torch.no_grad():
            output = model(seq)
        idx = torch.argmax(output, dim=1).item()
        generated.append(idx_to_word[idx])

    return " ".join(generated)

print(generate_words(model, ["artificial", "intelligence", "is", "transforming", "modern"], 30))


artificial intelligence is transforming modern training intelligent systems. large datasets help models learn complex patterns. deep learning uses multi layer neural network requires optimization techniques. gradient descent minimizes the loss function. systems. large datasets help
